### Fehlerfrei puzzeln – Lösung Schritt für Schritt

*30. Bundeswettbewerb Informatik (2011), 1. Runde, Aufgabe 3*

Im Taschenbuch der Algorithmen wurde folgendes Legespiel vorgestellt. Auf einem Brett liegen quadratische Spielsteine, deren vier Seiten je eine Farbe haben. Man darf die Steine auf andere der 12 · 8 quadratischen Plätze auf dem Brett legen. Dabei darf man die Steine aber nicht drehen – ihre Orientierung muss unverändert bleiben.

Liegen zwei verschiedenfarbige Seiten zweier Steine nebeneinander, erhält man einen Strafpunkt. Für eine gegebene Anfangsanordnung wurde eine neue Platzierung mit nur einem Strafpunkt vorgeschlagen (Bild im Aufgabenblatt).

#### Aufgabe 3

Schreibe ein Programm, das für solche Rätsel möglichst gute Lösungen findet und diese ausgibt.

Ist die oben gezeigte Platzierung optimal oder gibt es auch eine ohne Strafpunkte?

**Zu den Daten:** Die Datei `beispieldaten/anfangsanordnung_01.txt` beschreibt die 96 Steine des Bildes. Eine Formatbeschreibung gibt es nicht (die Hinweise klären es, siehe Schritt 1).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Kombinatorische Optimierung:** Aus riesig vielen Anordnungen (`96!` hat 150 Stellen) die mit den wenigsten Strafpunkten finden. Alle durchzuprobieren ist unmöglich.
- **Kostenfunktion und lokale Änderung:** Man bewertet eine Anordnung mit einer Zahl (hier: Strafpunkte). Beim Tauschen zweier Steine ändern sich nur die Kanten an diesen beiden Steinen, die Differenz lässt sich schnell ausrechnen.
- **Simulated Annealing (Heuristik):** Zufälliges Tauschen; Verbesserungen nimmt man immer an, Verschlechterungen mit einer Wahrscheinlichkeit, die mit sinkender „Temperatur“ abnimmt. Garantiert nichts, kann aber gute Ergebnisse liefern.
- **Backtracking mit Einschränkung (Constraint Propagation):** Für jedes Feld merkt man sich die **Menge der noch möglichen Steine**. Legt man einen Stein fest, schränkt das die Mengen der Nachbarfelder ein, und deren Einschränkung wirkt wieder auf deren Nachbarn. Man belegt immer das Feld mit der **kleinsten Menge** zuerst und beginnt nach zu langer Suche **neu mit anderer Zufallsreihenfolge**.
- **Zählargument:** Was an den Seiten nicht zusammenpasst, muss am Brettrand liegen. Das liefert eine notwendige Bedingung.

**Nützliche Funktionen und Bibliotheken**
- `set`: Mengen mit `&` (Schnitt), `|` (Vereinigung), `-` (Differenz), `len`, `sorted`.
- `dict` mit Tupeln als Schlüssel, z. B. `index[(seite, farbe)]`; `dict.setdefault`.
- `random.Random(zahl)`, `rnd.sample`, `rnd.shuffle`, `math.exp`: reproduzierbarer Zufall und die Annahmewahrscheinlichkeit.
- `divmod(p, breite)`: aus der Feldnummer Zeile und Spalte berechnen.
- `collections.Counter`: Farben zählen.

## Die Idee in Worten

Das Brett hat 8 Zeilen und 12 Spalten, wir nummerieren die Felder **zeilenweise von 0 an**: Feld `p` liegt in Zeile `p // 12` und Spalte `p % 12`. Ein **Brett** ist eine Liste mit 96 Steinen, ein **Stein** ein Tupel `(oben, rechts, unten, links)` aus Farbnamen.

1. **Einlesen und Format klären:** Welche Zahl ist x, welche y, und welche Farbe liegt an welcher Seite?
2. **Strafpunkte zählen:** Jedes Paar benachbarter Felder, deren Seiten verschiedene Farben haben, gibt einen Strafpunkt. Das Brett hat `8 · 11 + 12 · 7 = 172` solche Paare.
3. **Vorüberlegung:** Kann es überhaupt ein Brett ohne Strafpunkte geben? (Notwendige Bedingung durch Zählen.)
4. **Simulated Annealing** wie in den Lösungshinweisen: wie weit kommt man damit?
5. **Backtracking mit Mengen und Propagation:** findet eine Anordnung **ohne** Strafpunkte.
6. **Prüfen, ausgeben, an weiteren Rätseln testen.**
7. **Zum Schluss:** Dasselbe als Constraint-Modell in MiniZinc.

## Schritt 1: Einlesen und das Dateiformat klären

Jede Zeile der Datei beschreibt einen Stein: zwei Zahlen und vier Farben. Die erste Zahl liegt zwischen 1 und 12, die zweite zwischen 1 und 8: erst **Spalte `x`**, dann **Zeile `y`**. Das sind die Anfangsplätze der Steine. Die Reihenfolge der Farben steht nicht in der Datei. Die Lösungshinweise suchen dafür einen Stein, der nur einmal vorkommt: den mit drei grünen und einer orangen Seite. In der Anfangsanordnung liegt er auf Platz (2|3) und heißt `Green Green Green Orange`. Die orange Seite ist im Bild links, also ist die **letzte** Farbe die linke. Mit weiteren solchen Überlegungen ergibt sich: **Oben, Rechts, Unten, Links** (das nehmen wir von den Hinweisen).

Wir lesen die Steine so ein, dass der Stein vom Platz `(x|y)` in der Liste an Stelle `(y - 1) · 12 + (x - 1)` steht. So entsteht gleich das Brett der Anfangsanordnung.

In [ ]:
BREITE = 12
HOEHE = 8
OBEN, RECHTS, UNTEN, LINKS = 0, 1, 2, 3


def lies_steine(dateiname):
    brett = [None] * (BREITE * HOEHE)
    with open(dateiname, encoding='utf-8-sig') as f:
        for zeile in f:
            teile = zeile.split()
            if len(teile) == 6:
                x, y = int(teile[0]), int(teile[1])
                brett[(y - 1) * BREITE + (x - 1)] = tuple(teile[2:])
    return brett


anfang = lies_steine('beispieldaten/anfangsanordnung_01.txt')
print('Steine:', len(anfang), ' verschiedene Muster:', len(set(anfang)))
print('Feld 0 (Platz 1|1):', anfang[0])
print('Platz 2|3 (Feld %d):' % (2 * BREITE + 1), anfang[2 * BREITE + 1])
print('Steine mit drei grünen und einer orangen Seite:',
      [s for s in anfang if sorted(s) == ['Green', 'Green', 'Green', 'Orange']])

Steine: 96  verschiedene Muster: 81
Feld 0 (Platz 1|1): ('Orange', 'Yellow', 'Green', 'Orange')
Platz 2|3 (Feld 25): ('Green', 'Green', 'Green', 'Orange')
Steine mit drei grünen und einer orangen Seite: [('Green', 'Green', 'Green', 'Orange')]


Es gibt nur 81 verschiedene Muster für 96 Steine, also kommen manche Steine mehrfach vor. Der Stein mit drei grünen und einer orangen Seite ist wirklich einzigartig und liegt auf Platz (2|3), das stützt die Zuordnung der Hinweise.

## Schritt 2: Strafpunkte zählen und das Brett anzeigen

`strafpunkte` geht alle Felder durch und vergleicht, falls es sie gibt, den rechten Nachbarn (rechte Seite gegen linke Seite des Nachbarn) und den unteren Nachbarn (untere Seite gegen obere Seite). So wird jedes Nachbarpaar genau einmal gezählt. `zeige` druckt jeden Stein als vier Anfangsbuchstaben (oben, rechts, unten, links).

In [ ]:
def strafpunkte(brett):
    punkte = 0
    for p in range(BREITE * HOEHE):
        zeile, spalte = divmod(p, BREITE)
        if spalte < BREITE - 1 and brett[p][RECHTS] != brett[p + 1][LINKS]:
            punkte += 1
        if zeile < HOEHE - 1 and brett[p][UNTEN] != brett[p + BREITE][OBEN]:
            punkte += 1
    return punkte


def zeige(brett):
    for zeile in range(HOEHE):
        felder = brett[zeile * BREITE:(zeile + 1) * BREITE]
        print(' '.join(''.join(farbe[0] for farbe in stein) for stein in felder))


print('Nachbarpaare:', (BREITE - 1) * HOEHE + BREITE * (HOEHE - 1))
print('Strafpunkte der Anfangsanordnung:', strafpunkte(anfang))
zeige(anfang)

Nachbarpaare: 172
Strafpunkte der Anfangsanordnung: 127
OYGO YYGO BGYG YYYO GOOB OBBB OGYB BYGO BBBY OBOG YGGB GGYO
GOYB GBOY GGBO OYYY GGYG GYGB OOBG YYGY YGBB GBYO GOOO BYYG
GGYO GGGO YGYO OGGY YOOB YBBY BBOO YBGO YYBG YYBB GBOO YYBY
YOOY BGYB YBOY GBBB BOBG OBOO GYYB OBBO YBBB YBBY BOGB GGOB
BYBB OYGY OBBY GBYG OOYB OGOB BYBB YBOO BOOY BOGB OYBY BOYB
OBOG OBBG BBOO GGBO BOBG BOGB GGGB YBBY BOGO OYOB BBYG OGGB
YYYO YYBG YOBB BGOG BOYG OOYG BOYO OYGG GYGY BBOG BGYB YOBY
OOGY GGBO BBOB GOGO YOGO YOGG YOBG GYYG BOOG BYYO BBBY OBGG


Die Anfangsanordnung hat **127 Strafpunkte** bei 172 Nachbarpaaren, fast jedes Paar passt nicht (bei 4 Farben passt ein zufälliges Paar mit Wahrscheinlichkeit 1/4). Die „Farbbuchstaben“ sind `O`range, `Y`ellow, `G`reen und `B`lue.

## Schritt 3: Vorüberlegung durch Zählen

Gibt es überhaupt eine Chance auf 0 Strafpunkte? Auf einer waagerechten Kante trifft eine **rechte** Seite auf eine **linke** Seite derselben Farbe. Gibt es von einer Farbe mehr rechte als linke Seiten, müssen die überzähligen rechten Seiten **am rechten Brettrand** liegen (dort ist kein Nachbar). Der rechte Rand hat nur 8 Felder, der linke auch. Das passt nur, wenn die Überschüsse klein genug sind. Genauso senkrecht (Rand oben und unten: je 12 Felder).

Hinweise zu diesem Gedanken: Wenn z. B. 17 Steine an der unteren Seite ein gelbes Dreieck haben, aber 20 Steine an der oberen Seite, müssen mindestens drei davon am oberen Rand liegen.

In [ ]:
from collections import Counter


def ueberschuss(steine, seite_a, seite_b):
    """Für jede Farbe: Anzahl der Seiten a minus Anzahl der Seiten b."""
    a = Counter(stein[seite_a] for stein in steine)
    b = Counter(stein[seite_b] for stein in steine)
    return {farbe: a[farbe] - b[farbe] for farbe in sorted(a)}


waag = ueberschuss(anfang, RECHTS, LINKS)
senk = ueberschuss(anfang, UNTEN, OBEN)
print('rechts minus links:', waag)
print('unten minus oben:  ', senk)
print('Überzählige rechte Seiten (müssen an den rechten Rand):', sum(max(0, d) for d in waag.values()), 'von 8 Randfeldern')
print('Überzählige linke Seiten (müssen an den linken Rand):  ', sum(max(0, -d) for d in waag.values()), 'von 8 Randfeldern')
print('Überzählige untere Seiten (müssen an den unteren Rand):', sum(max(0, d) for d in senk.values()), 'von 12 Randfeldern')
print('Überzählige obere Seiten (müssen an den oberen Rand):  ', sum(max(0, -d) for d in senk.values()), 'von 12 Randfeldern')

rechts minus links: {'Blue': -1, 'Green': -3, 'Orange': 0, 'Yellow': 4}
unten minus oben:   {'Blue': 1, 'Green': 1, 'Orange': -1, 'Yellow': -1}
Überzählige rechte Seiten (müssen an den rechten Rand): 4 von 8 Randfeldern
Überzählige linke Seiten (müssen an den linken Rand):   4 von 8 Randfeldern
Überzählige untere Seiten (müssen an den unteren Rand): 2 von 12 Randfeldern
Überzählige obere Seiten (müssen an den oberen Rand):   2 von 12 Randfeldern


Die Überschüsse passen in die Ränder (4 von 8 und 2 von 12). **Die Zählung schließt 0 Strafpunkte also nicht aus.** Sie ist aber nur eine notwendige Bedingung und beweist nicht, dass es eine fehlerfreie Anordnung gibt. Bei einem anderen Rätsel könnte sie das Ergebnis aber sofort widerlegen (zum Beispiel, wenn von einer Farbe 10 rechte Seiten zu viel wären).

## Schritt 4: Simulated Annealing (Verfahren der Lösungshinweise)

Die Hinweise schlagen vor: Zwei Steine zufällig vertauschen. Wird die Anordnung nicht schlechter, bleibt der Tausch. Sonst bleibt er nur mit Wahrscheinlichkeit `e^(-d / T)`, wobei `d` die Zunahme der Strafpunkte und `T` die „Temperatur“ ist. Sie startet bei `T0 = 8` (größtmögliche Verschlechterung durch einen Tausch) und wird alle `k` Schritte mit einer Zahl `o < 1` multipliziert, `T(n) = T0 · o^(n // k)`. Je kälter, desto seltener wird eine Verschlechterung angenommen.

**Schneller Strafpunkt-Unterschied:** Beim Tausch ändern sich nur die Kanten an den beiden Feldern. Wir sammeln diese Kanten in einer Menge (so zählt eine gemeinsame Kante, wenn die Felder Nachbarn sind, nur einmal) und zählen vor und nach dem Tausch nur diese. `KANTEN_VON[p]` enthält die Kanten an Feld `p` als Tupel `(feld_a, feld_b, seite_a, seite_b)`.

In [ ]:
import math
import random

KANTEN_VON = [[] for _ in range(BREITE * HOEHE)]
for p in range(BREITE * HOEHE):
    zeile, spalte = divmod(p, BREITE)
    if spalte < BREITE - 1:
        kante = (p, p + 1, RECHTS, LINKS)
        KANTEN_VON[p].append(kante)
        KANTEN_VON[p + 1].append(kante)
    if zeile < HOEHE - 1:
        kante = (p, p + BREITE, UNTEN, OBEN)
        KANTEN_VON[p].append(kante)
        KANTEN_VON[p + BREITE].append(kante)


def falsche_kanten(brett, kanten):
    return sum(1 for a, b, sa, sb in kanten if brett[a][sa] != brett[b][sb])


def annealing(start, schritte, t0, o, k, seed):
    rnd = random.Random(seed)
    brett = list(start)
    punkte = strafpunkte(brett)
    bester = punkte
    for n in range(schritte):
        if punkte == 0:
            break
        temperatur = t0 * o ** (n // k)
        a, b = rnd.sample(range(len(brett)), 2)
        kanten = set(KANTEN_VON[a] + KANTEN_VON[b])
        vorher = falsche_kanten(brett, kanten)
        brett[a], brett[b] = brett[b], brett[a]
        d = falsche_kanten(brett, kanten) - vorher
        if d <= 0 or rnd.random() < math.exp(-d / temperatur):
            punkte += d                              # Tausch behalten
            bester = min(bester, punkte)
        else:
            brett[a], brett[b] = brett[b], brett[a]  # Tausch rückgängig
    return brett, punkte


for seed in (1, 2, 3):
    ergebnis, p = annealing(anfang, 300000, 8, 0.96, 2500, seed)
    assert p == strafpunkte(ergebnis)
    print('Seed', seed, ': Strafpunkte nach 300000 Tauschversuchen:', p)

Seed 1 : Strafpunkte nach 300000 Tauschversuchen: 5
Seed 2 : Strafpunkte nach 300000 Tauschversuchen: 4
Seed 3 : Strafpunkte nach 300000 Tauschversuchen: 7


**Ergebnis:** Mit diesen Einstellungen (300 000 Tauschversuche, `T0 = 8`, `o = 0.96`, `k = 2500`) kommen wir auf wenige Strafpunkte, aber **nicht auf 0**. Mit anderen Einstellungen, anderen Startwerten und vielen Millionen Schritten hätten wir mehr Chancen: Die Hinweise schreiben, dass man damit sogar die Lösung ohne Strafpunkte findet, nennen aber zugleich, dass `o` und `k` durch Probieren zu bestimmen sind. Der Nachteil: Das Verfahren **garantiert nichts**. Hört es mit einigen Strafpunkten auf, wissen wir nicht, ob es 0 gibt.

Ein Tausch mit anderen Steinen hilft kaum noch, wenn fast alles passt. Für Rätsel wie dieses gibt es ein Verfahren, das **systematisch** sucht und uns nicht auf den Zufall verlässt.

## Schritt 5: Backtracking mit Mengen von Kandidaten

Statt ein volles Brett zu verändern, bauen wir es **Feld für Feld** auf und lassen **kein Strafpunkt entstehen**: Jede Kante muss passen. Das ist ein Backtracking-Verfahren. Das einfachste (Felder zeilenweise füllen und nur nach links und oben vergleichen) bleibt aber in dieser Aufgabe hängen: In unseren Versuchen kam es auch nach mehreren Millionen Schritten nicht ans Ziel. Hier die Verbesserung in drei Teilen.

**Teil 1: Für jedes Feld die Menge der noch möglichen Steine.** `dom[p]` ist die Menge der Steinnummern (0 bis 95), die auf Feld `p` noch passen könnten. Am Anfang kommen alle in Frage.

**Teil 2: Propagation.** Hat Feld `p` die Menge `D`, dann kommen für den Nachbarn `q` nur Steine in Frage, deren Seite zu einer Farbe aus den Steinen in `D` passt. Ist die Menge von `q` dadurch kleiner geworden, prüfen wir von `q` aus wieder dessen Nachbarn, und so weiter. Außerdem gilt: Ist für ein Feld nur ein Stein übrig, scheidet dieser Stein bei **allen anderen Feldern** aus (jeder Stein liegt nur einmal). Wird eine Menge leer, geht es auf diesem Weg nicht weiter.

**Teil 3: Suche.** Wir wählen das Feld mit der **kleinsten Menge** (mehr als ein Kandidat), probieren seine Steine in zufälliger Reihenfolge aus, propagieren und machen rekursiv weiter. Klappt es nicht, gehen wir zurück (Backtracking). Zählen wir mehr als `limit` Knoten, brechen wir ab und **fangen mit einem neuen Zufalls-Seed neu an**. Mit einer zu unglücklichen Reihenfolge steckt man sonst ewig fest.

**Die Funktionen sind für beliebige Rätsel geschrieben:** Sie bekommen die Liste der Steine und die Maße des Bretts.

In [ ]:
def bereite_vor(steine, breite, hoehe):
    # index[(seite, farbe)] = Menge der Steine, die an dieser Seite diese Farbe haben
    index = {}
    for i, stein in enumerate(steine):
        for seite in range(4):
            index.setdefault((seite, stein[seite]), set()).add(i)
    # nachbarn[p] = Liste von (Nachbarfeld, meine Seite, Seite des Nachbarn)
    nachbarn = []
    for p in range(breite * hoehe):
        zeile, spalte = divmod(p, breite)
        liste = []
        if spalte < breite - 1:
            liste.append((p + 1, RECHTS, LINKS))
        if spalte > 0:
            liste.append((p - 1, LINKS, RECHTS))
        if zeile < hoehe - 1:
            liste.append((p + breite, UNTEN, OBEN))
        if zeile > 0:
            liste.append((p - breite, OBEN, UNTEN))
        nachbarn.append(liste)
    return index, nachbarn


def propagiere(dom, start, steine, index, nachbarn):
    """Schränkt die Mengen ein. Gibt False zurück, wenn eine Menge leer wird."""
    warteschlange = list(start)
    while warteschlange:
        p = warteschlange.pop()
        D = dom[p]
        if not D:
            return False
        for q, meine_seite, seite_nachbar in nachbarn[p]:
            erlaubt = set()
            for farbe in {steine[i][meine_seite] for i in D}:
                erlaubt |= index[(seite_nachbar, farbe)]
            neu = dom[q] & erlaubt
            if len(neu) < len(dom[q]):
                if not neu:
                    return False
                dom[q] = neu
                warteschlange.append(q)
        if len(D) == 1:                                  # dieser Stein liegt auf p, also nirgends sonst
            for q in range(len(dom)):
                if q != p and D <= dom[q]:
                    dom[q] = dom[q] - D
                    if not dom[q]:
                        return False
                    warteschlange.append(q)
    return True


print('Funktionen definiert')

Funktionen definiert


`suche` ist die Rekursion. Sie gibt drei verschiedene Antworten: die fertigen Mengen (alle haben genau einen Stein), `False` (in diesem Teil des Suchbaums gibt es keine Lösung) oder `None` (zu viele Knoten, Abbruch zum Neustart). `loese_puzzle` startet bis zu `versuche` Läufe mit den Seeds 0, 1, 2, ... und liefert das Brett zusammen mit dem Protokoll der Läufe. Antwortet ein Lauf mit `False`, hat die Suche den **ganzen** Suchbaum abgesucht: Das Rätsel ist **unlösbar**, weitere Läufe sind überflüssig.

In [ ]:
def suche(dom, d):
    d['knoten'] += 1
    if d['knoten'] > d['limit']:
        return None
    offen = [p for p in range(len(dom)) if len(dom[p]) > 1]
    if not offen:
        return dom
    p = min(offen, key=lambda f: len(dom[f]))           # Feld mit den wenigsten Kandidaten
    kandidaten = sorted(dom[p])
    d['rnd'].shuffle(kandidaten)
    for i in kandidaten:
        neu = list(dom)                                 # Kopie der Mengenliste (die Mengen selbst ändert propagiere nie)
        neu[p] = {i}
        if propagiere(neu, [p], d['steine'], d['index'], d['nachbarn']):
            ergebnis = suche(neu, d)
            if ergebnis is None:
                return None
            if ergebnis:
                return ergebnis
    return False


def loese_puzzle(steine, breite, hoehe, limit=3000, versuche=50):
    index, nachbarn = bereite_vor(steine, breite, hoehe)
    protokoll = []
    for seed in range(versuche):
        d = {'steine': steine, 'index': index, 'nachbarn': nachbarn, 'rnd': random.Random(seed),
             'limit': limit, 'knoten': 0}
        dom = [set(range(len(steine))) for _ in steine]
        ergebnis = suche(dom, d)
        protokoll.append((seed, d['knoten']))
        if ergebnis:
            return [steine[next(iter(m))] for m in ergebnis], protokoll
        if ergebnis is False:
            return None, protokoll                       # Suchbaum vollständig abgesucht: unlösbar
    return None, protokoll


print('Funktionen definiert')

Funktionen definiert


Beachte `neu = list(dom)`: Die Kopie der Liste ist **flach**, die Mengen darin sind dieselben. Das ist hier in Ordnung, weil `propagiere` nie eine Menge ändert (mit `&=` oder `.remove`), sondern immer eine **neue** Menge in die Liste einträgt (`dom[q] = neu`). So kann der Zustand vor dem Ausprobieren nicht beschädigt werden.

Jetzt auf das Beispiel anwenden. Die Steine geben wir in beliebiger Reihenfolge hinein (die Anfangsanordnung ist als Liste dafür gut genug, ihre Reihenfolge spielt keine Rolle):

In [ ]:
ergebnis, protokoll = loese_puzzle(anfang, BREITE, HOEHE)
print('Läufe (Seed, Knoten):', protokoll)
print('Strafpunkte:', strafpunkte(ergebnis))
zeige(ergebnis)

Läufe (Seed, Knoten): [(0, 3001), (1, 2533)]
Strafpunkte: 0
OBBG BGYB OOBG GGGO BOOG YYYO GYGY YBOY OGYB YOGG GOOO GGYO
BOBG YYYO BBBY GYGB OOGY YBGO GBBB OBBB YGGB GGYG OBOG YYBB
BOYO YGYO BOYG GBYO GOOB GBOO BOGB BBOO GOYB YBOO OOYB BBOO
YBBB YGBB YOBG YOGO OYGO OBBY GYYB OYGY YYGY OYBY YBBY OYOB
BGYG BBOG BGYB GBYG GGGB BYYG YOOY GGBO GYYG BBBY BYBB OYYY
YYBG OGGY YYBG YYBY GBOY YOOB OBBO BYBB YBBY BOYB BYYO YBBY
BBOB GGOB BBYG BOGB OBOO OGGB BGOG BOBG BYGO YOBY YYGO BOOY
OGOB OBGG YOBB GGBO OOYG GGYO OBOG BOGB GOGO BOGO GGBO OYGG


**Das Rätsel hat eine Lösung ohne Strafpunkte.** Damit ist die Platzierung auf dem Aufgabenblatt mit einem Strafpunkt **nicht optimal**. Der erste Lauf mit Seed 0 bricht nach 3000 Knoten ab, der zweite findet die Lösung. Der Aufwand ist klein, im Vergleich zum Millionen-Schritte-Annealing eine gute Verbesserung.

Die Anordnung oben ist eine von vielen. Ein anderer Seed oder anderes `limit` führt zu einer anderen (zum Beispiel findet MiniZinc im letzten Abschnitt eine andere).

## Schritt 6: Gegenprobe und Ausgabe

Eine unabhängige Prüfung: Das Ergebnis enthält **genau die Steine der Eingabe** (jeder Stein einmal, nicht gedreht) und hat 0 Strafpunkte. `als_text` schreibt die Anordnung in dem Format der Eingabedatei: Platz `x y` und die vier Farben.

In [ ]:
def pruefe(brett, steine):
    return sorted(brett) == sorted(steine) and strafpunkte(brett) == 0


def als_text(brett):
    zeilen = []
    for p, stein in enumerate(brett):
        zeile, spalte = divmod(p, BREITE)
        zeilen.append('%2d %2d %s' % (spalte + 1, zeile + 1, ' '.join(stein)))
    return '\n'.join(zeilen)


print('Gültig und fehlerfrei:', pruefe(ergebnis, anfang))
print('Gültig und fehlerfrei (Anfangsanordnung):', pruefe(anfang, anfang))
ausgabe = als_text(ergebnis).splitlines()
print('\n'.join(ausgabe[:4]), '...', sep='\n')

Gültig und fehlerfrei: True
Gültig und fehlerfrei (Anfangsanordnung): False
 1  1 Orange Blue Blue Green
 2  1 Blue Green Yellow Blue
 3  1 Orange Orange Blue Green
 4  1 Green Green Green Orange
...


## Schritt 7: Weitere Rätsel – und ein unlösbares

Das Programm soll nicht nur auf das eine Beispiel passen. Zum Testen brauchen wir Rätsel, von denen wir **wissen**, dass sie lösbar sind: `zufallspuzzle` würfelt zuerst die Farben aller Kanten (auch am Rand) und baut daraus passende Steine, so entsteht ein fehlerfreies Brett. Danach mischen wir die Steine. Das Verfahren muss ein Brett ohne Strafpunkte finden (nicht unbedingt das, aus dem das Rätsel entstand). Dazu drei Rätsel (6 × 5 mit 3 Farben, 6 × 5 mit 4 Farben, 8 × 6 mit 3 Farben) und ein Beispiel, das offensichtlich **unlösbar** ist. Größere Zufallsrätsel mit lauter verschiedenen Steinen (zum Beispiel 12 × 8) sind für unser Programm viel schwerer: Viele Läufe brechen ab, weil es dort nur wenige Lösungen gibt. Das Beispiel der Aufgabe hat nur 81 verschiedene Muster für 96 Steine (manche Steine gibt es mehrfach), das macht es leichter.

Die Funktion `strafpunkte` kennt die Maße über die globalen Zahlen `BREITE` und `HOEHE`. Für die Tests mit anderen Maßen ändern wir diese Zahlen kurz (und setzen sie danach zurück).

In [ ]:
def zufallspuzzle(breite, hoehe, farben, seed):
    rnd = random.Random(seed)
    waagerecht = [[rnd.choice(farben) for _ in range(breite + 1)] for _ in range(hoehe)]     # Farbe der Kante links von Spalte x
    senkrecht = [[rnd.choice(farben) for _ in range(breite)] for _ in range(hoehe + 1)]      # Farbe der Kante über Zeile y
    steine = []
    for y in range(hoehe):
        for x in range(breite):
            steine.append((senkrecht[y][x], waagerecht[y][x + 1], senkrecht[y + 1][x], waagerecht[y][x]))
    rnd.shuffle(steine)
    return steine


def teste(breite, hoehe, farben, seed):
    global BREITE, HOEHE
    alt = (BREITE, HOEHE)
    BREITE, HOEHE = breite, hoehe
    try:
        steine = zufallspuzzle(breite, hoehe, farben, seed)
        brett, protokoll = loese_puzzle(steine, breite, hoehe)
        return steine, brett, protokoll, (strafpunkte(brett) if brett else None)
    finally:
        BREITE, HOEHE = alt


FARBEN5 = ['Orange', 'Yellow', 'Green', 'Blue', 'Red']
for breite, hoehe, farben, seed in [(6, 5, FARBEN5[:3], 3), (6, 5, FARBEN5[:4], 1), (8, 6, FARBEN5[:3], 2)]:
    steine, brett, protokoll, p = teste(breite, hoehe, farben, seed)
    print('%d x %d, %d Farben, Seed %d: Läufe %d, Strafpunkte %s, Steine unverändert: %s' % (
        breite, hoehe, len(farben), seed, len(protokoll), p, sorted(brett) == sorted(steine)))
    assert p == 0 and sorted(brett) == sorted(steine)

6 x 5, 3 Farben, Seed 3: Läufe 1, Strafpunkte 0, Steine unverändert: True
6 x 5, 4 Farben, Seed 1: Läufe 1, Strafpunkte 0, Steine unverändert: True
8 x 6, 3 Farben, Seed 2: Läufe 1, Strafpunkte 0, Steine unverändert: True


In [ ]:
# Unlösbar: Alle Steine haben vier verschiedene Farben an den Seiten, kein Stein passt neben einen anderen
unloesbar = [('Orange', 'Yellow', 'Green', 'Blue'),
             ('Yellow', 'Green', 'Blue', 'Orange'),
             ('Green', 'Blue', 'Orange', 'Yellow'),
             ('Blue', 'Orange', 'Yellow', 'Green')]
brett, protokoll = loese_puzzle(unloesbar, 2, 2)
print('Ergebnis:', brett, ' Läufe:', protokoll)

Ergebnis: None  Läufe: [(0, 1)]


Bei diesem 2 × 2-Rätsel passt kein Stein neben einen anderen, die Suche stellt das mit einem einzigen Knoten fest und gibt `None` zurück, und das schon im ersten Lauf (`protokoll` hat nur einen Eintrag): Das Rätsel ist **unlösbar**, wir müssen nicht weiter probieren. (Ein `None` nach allen `versuche` Läufen bedeutet dagegen nur „nichts gefunden“, nicht „unlösbar“; das erkennt man an der Länge von `protokoll`.) Das ist ein Vorteil gegenüber dem Annealing, das nie „unlösbar“ sagen kann. Bei einem **knapp** unlösbaren Rätsel (alles passt bis auf eine Kante) würde die Suche allerdings sehr lang laufen. Dann hilft das Annealing wieder als Notlösung: Es liefert wenigstens ein Brett mit wenigen Strafpunkten.

## Für Neugierige: Warum Neustarts?

Die Zahl der Knoten bis zur Lösung hängt stark von der zufälligen Reihenfolge ab (**„heavy tail“**: meistens nach wenigen tausend Knoten gefunden, manchmal erst nach sehr vielen). Wir haben das Beispiel mit mehr Seeds ausprobiert:

In [ ]:
index_b, nachbarn_b = bereite_vor(anfang, BREITE, HOEHE)
tabelle = []
for seed in range(6):
    d = {'steine': anfang, 'index': index_b, 'nachbarn': nachbarn_b, 'rnd': random.Random(seed),
         'limit': 3000, 'knoten': 0}
    r = suche([set(range(96)) for _ in range(96)], d)
    tabelle.append((seed, 'gefunden' if r else 'Abbruch nach 3000'))
print(tabelle)

[(0, 'Abbruch nach 3000'), (1, 'gefunden'), (2, 'Abbruch nach 3000'), (3, 'Abbruch nach 3000'), (4, 'Abbruch nach 3000'), (5, 'Abbruch nach 3000')]


Nur manche Läufe finden die Lösung innerhalb von 3000 Knoten (hier ist es Seed 1). Deshalb ist es besser, **oft mit einem kleinen `limit` neu zu starten**, als einen einzigen Lauf ewig laufen zu lassen. Ein günstiges `limit` bestimmt man durch Probieren, ähnlich wie `o` und `k` beim Annealing.

## Zum Schluss: Geht das auch mit MiniZinc?

Ja, und das Modell ist kürzer als unser Python-Programm. Das Rätsel ist ein **Constraint-Problem**: Wir beschreiben nur, was gelten muss, und der Solver sucht eine Belegung. Die Mengen-Propagation, die wir in Python von Hand geschrieben haben, erledigt er selbst.

**Die Variablen:** `s[y, x]` ist die Nummer des Steins auf Feld `(y, x)`, ein Wert von 1 bis 96 (**Variablen als Index**).

**Die Bedingungen:**
- `alldifferent`: Jeder Stein liegt auf genau einem Feld.
- Für jedes waagerechte Nachbarpaar: Die Farbe an der rechten Seite von Stein `s[y, x]` (`farbe[s[y, x], 2]`) ist gleich der Farbe an der linken Seite von `s[y, x+1]`. Der Index `s[y, x]` ist selbst eine Variable, MiniZinc kann das (es heißt **`element`-Bedingung**).
- Genauso senkrecht (untere gegen obere Seite).

Es gibt **kein** Ziel (`solve satisfy`), denn wir suchen ein fehlerfreies Brett. Nur die fehlerfreie Anordnung zählt; die Strafpunkte werden nicht gezählt, sondern ganz verboten. Möchte man auch dann ein bestes Brett, wenn es kein fehlerfreies gibt, braucht man ein Ziel (`minimize` Anzahl falscher Kanten) und das Modell wird viel schwerer.

Die Daten (`.dzn`) erzeugt eine Python-Zelle `als_dzn` aus den Steinen: Die Farben werden zu den Zahlen 1 bis 4 (alphabetisch: Blue, Green, Orange, Yellow). Auszug:

```minizinc
B = 12;
H = 8;
farbe = array2d(1..96, 1..4, [3, 4, 2, 3, 4, 4, 2, 3, 1, 2, 4, 2] ++ ...);
```

```minizinc
include "alldifferent.mzn";

int: B;                                      % Breite des Bretts
int: H;                                      % Hoehe des Bretts
int: N = B * H;                              % Anzahl der Steine
array[1..N, 1..4] of int: farbe;             % farbe[i, k]: Farbe (1..4) von Stein i an Seite k (oben, rechts, unten, links)

array[1..H, 1..B] of var 1..N: s;            % s[y, x]: Nummer des Steins auf Feld (y, x)

constraint alldifferent([s[y, x] | y in 1..H, x in 1..B]);                          % jeder Stein genau einmal
constraint forall(y in 1..H, x in 1..B-1)(farbe[s[y, x], 2] = farbe[s[y, x+1], 4]);  % rechte Seite = linke Seite des Nachbarn
constraint forall(y in 1..H-1, x in 1..B)(farbe[s[y, x], 3] = farbe[s[y+1, x], 1]);  % untere Seite = obere Seite darunter

solve satisfy;
output ["\(s)\n"];
```

**Ausgabe** (Gecode), erste Zeile des Bretts, gekürzt (Nummern der Steine, gezählt wie in der Anfangsanordnung: zeilenweise von 1 bis 96):

```
[68, 83, 76, 90, 23, 89, 44, 54, 93, 35, 55, 59, ...]
```

**Messung und Vergleich mit Python.** Wir haben das Modell auf das Beispiel (das einzige) angewendet:
- **Gecode** findet nach weniger als einer Sekunde ein fehlerfreies Brett. Stein `i` ist `anfang[i - 1]` in unserer Python-Liste. Wir haben mit unserer Funktion `pruefe` nachgerechnet: Jeder Stein kommt genau einmal vor, 0 Strafpunkte.
- Das Brett ist **ein anderes** als das von Python gefundene. Beide sind richtig: Es gibt sehr viele fehlerfreie Anordnungen, welche der Solver zuerst findet, hängt von der Suche ab.
- **Chuffed** findet in einer Minute keine Lösung, **CP-SAT** braucht rund eine halbe Minute. Für dieses Problem eignet sich also die Suche von Gecode am besten. Das ist eine **Grenze** des Verfahrens: Der Solver wählt die Suchreihenfolge selbst, und bei manchen Solvern ist sie bei diesem Rätsel unglücklich (ähnlich wie bei unserem Python-Backtracking ohne Mengen).

Für Rätsel, die **nicht fehlerfrei** lösbar sind, ist MiniZinc hier keine Hilfe (es sagt `UNSATISFIABLE`, liefert aber kein bestes Brett). Dafür bleiben Annealing und andere Heuristiken die richtige Wahl.